In [ ]:
from trustutils import run
run.TRUST_parameters()
run.introduction('MMS Exponential+Trig 3D', '2026')


In [ ]:
run.reset()

NB_PROCS = 2

case = run.addCase('.', 'MMS_ExpTrig3D.data', nbProcs=NB_PROCS)
case.partition()          # trust -partition MMS_ExpTrig3D.data 2 -> PAR_MMS_ExpTrig3D.data + DOM_000*.Zones
run.printCases()
run.runCases()
run.tablePerf()


In [ ]:
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# MMS_ExpTrig3D: b(x, y, 0) = sin(pi x) exp(-y), u = 0, eta = 0.01, with
# Neumann walls in x/z and thin Dirichlet layers at y = 0, 1.  Away from the
# walls the local decay rate is eta * |lap b| / b = eta * (pi^2 - 1); over the
# nb_pas_dt_max-capped run the probe ratios must follow it to a few percent.
eta = 0.01
rate = eta * (np.pi**2 - 1.0)

for name in ('SONDE_Y025', 'SONDE_Y050', 'SONDE_Y075'):
    t, b = read_son(case.dataFileName_, name)
    ratio = b[-1] / b[0]
    expected = np.exp(-rate * t[-1])
    err = abs(ratio - expected) / expected
    print(f"{name}: ratio={ratio:.4f} expected={expected:.4f} err={err:.3e}")
    assert err < 0.10, f"{name} decay error too large: {err:.3e}"
print("OK: exp-trig decay matches the local-rate prediction at all probes.")
